# Atividade: construindo uma árvore de decisão

Na semana passada você escreveu à mão as regras que separavam as frutas. Esta semana o algoritmo vai escolher as regras sozinho, e você vai implementá-lo.

A árvore de decisão faz uma sequência de perguntas de sim ou não sobre os atributos. Cada folha devolve uma classe. O que você precisa programar é como ela escolhe cada pergunta: medir a mistura de um grupo, comparar cortes candidatos e decidir quando parar.

O conjunto cresceu. Além de maçã, tangerina, laranja e limão, agora existem toranjas e limas, com poucos exemplos cada. Esse desequilíbrio é proposital e vai aparecer na parte de métricas.

Execute as células na ordem. Onde houver `...`, o código é seu; os comentários em volta indicam o que cada trecho precisa fazer.

## 1. Conhecer os dados

As colunas são as mesmas da semana passada: `mass`, `width`, `height`, `color_score` e a classe `fruit_name`.

In [ ]:
import pandas as pd

TRAIN_URL = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/frutas/train_extended.csv'
train = pd.read_csv(TRAIN_URL)

FEATURES = ['mass', 'width', 'height', 'color_score']
TARGET = 'fruit_name'

X_train = train[FEATURES]
y_train = train[TARGET]

print(f'{len(train)} frutas')
train[TARGET].value_counts()

Observe a contagem por classe antes de continuar. Duas classes têm menos de seis exemplos.

**Antes de programar, responda:** se um modelo respondesse sempre `orange`, qual seria sua acurácia? Isso o tornaria um bom classificador?

**Minha resposta:** ...

## 2. Medir a mistura de um grupo

O primeiro ingrediente é uma medida de impureza: um número que vale zero quando o grupo tem uma única classe e cresce conforme ele se mistura.

O índice de Gini foi o critério apresentado na aula:

$$G = 1 - \sum_i p_i^2$$

onde $p_i$ é a proporção de exemplos da classe $i$ no grupo. Se preferir usar a entropia, ou outro critério, fique à vontade: a construção da árvore é a mesma, muda apenas a função que mede a mistura. Registre qual você escolheu.

In [ ]:
def impurity(labels):
    """Recebe uma sequência de classes e devolve a impureza do grupo."""
    # Conte quantos exemplos há de cada classe, calcule a proporção de cada uma
    # e combine as proporções em um único número.
    ...

In [ ]:
# Confira com os números da aula antes de seguir.
puro = ['orange'] * 14
misturado = ['orange'] * 14 + ['mandarin'] * 4

print('grupo puro:      ', round(impurity(puro), 3))        # Gini: 0.0
print('14 laranjas e 4 tangerinas:', round(impurity(misturado), 3))  # Gini: 0.346
print('conjunto inteiro:', round(impurity(y_train), 3))

## 3. Encontrar o melhor corte

Um corte é uma pergunta do tipo `atributo <= limiar?`. Ele divide o grupo em dois: à esquerda ficam as frutas que respondem sim, à direita as que respondem não.

A árvore não tem intuição sobre frutas. Ela testa muitos cortes candidatos e fica com o melhor. Então precisamos de duas coisas: uma lista de candidatos e um jeito de comparar dois deles.

### Quais limiares testar

Para um atributo contínuo como a massa, existem infinitos limiares possíveis. Mas quase todos são equivalentes: cortar em 150,0 ou em 150,4 separa exatamente as mesmas frutas, se não houver nenhuma fruta entre esses dois valores. O que importa são os pontos onde a divisão realmente muda.

Então basta olhar os valores que aparecem no grupo. Se as massas presentes são 116, 118, 120 e 152, os cortes que produzem divisões diferentes estão entre valores consecutivos. Uma escolha comum é usar o ponto médio: 117, 119 e 136.

Usar diretamente os valores observados (`<= 116`, `<= 118`, ...) também funciona e é mais simples de escrever. A diferença aparece só nos limiares que a árvore vai imprimir no fim. Escolha uma das duas e registre qual usou.

### Como comparar dois cortes

Cada lado do corte é um grupo, e você já sabe medir a impureza de um grupo. Falta juntar os dois números em um só.

Não serve a média simples. Um corte que separa 1 fruta de 55 pode deixar o lado de 1 fruta perfeitamente puro, e isso não significa quase nada. Por isso cada lado pesa pelo seu tamanho:

$$\text{custo} = \frac{n_{esq} \cdot G_{esq} + n_{dir} \cdot G_{dir}}{n}$$

O melhor corte é o de **menor** custo, porque custo baixo significa lados pouco misturados.

### O que programar

Duas funções auxiliares, e depois a busca:

1. `candidate_thresholds(values)` — recebe a coluna de um atributo e devolve os limiares a testar;
2. `weighted_impurity(y_left, y_right)` — recebe os rótulos dos dois lados e devolve o custo;
3. `best_split(X, y)` — percorre todos os atributos e todos os limiares, e devolve o de menor custo.

Na busca, descarte os cortes que deixam um dos lados vazio: eles não dividem nada e sempre têm custo igual ao do grupo inteiro.

In [ ]:
def candidate_thresholds(values):
    """Recebe a coluna de um atributo e devolve os limiares a testar."""
    # Pegue os valores distintos, ordene-os e combine cada par de consecutivos.
    ...


def weighted_impurity(y_left, y_right):
    """Custo de um corte: a média das impurezas dos dois lados,
    ponderada pelo tamanho de cada um."""
    ...

In [ ]:
# Confira as auxiliares em um caso pequeno, onde dá para verificar na mão.
print(candidate_thresholds(pd.Series([116, 118, 120, 152])))

puro_esq = ['lemon'] * 4
misto_dir = ['orange'] * 14 + ['mandarin'] * 4
print(round(weighted_impurity(puro_esq, misto_dir), 3))  # (4 * 0 + 18 * 0.346) / 22

In [ ]:
def best_split(X, y):
    """Devolve (feature, threshold, custo) do melhor corte, ou None se nenhum
    corte dividir o grupo em dois lados não vazios."""
    best = None

    for feature in FEATURES:
        for threshold in candidate_thresholds(X[feature]):
            # 1. monte as duas máscaras: X[feature] <= threshold e o complemento
            # 2. se um dos lados ficar vazio, pule este limiar
            # 3. calcule o custo com weighted_impurity
            # 4. se for o menor custo visto até agora, guarde-o em best
            ...

    return best

In [ ]:
# O primeiro corte da árvore da aula foi color_score <= 0.825.
best_split(X_train, y_train)

Se o seu resultado foi outro, verifique a lista de limiares testados e a média ponderada antes de continuar. Pequenas diferenças no limiar são aceitáveis; o atributo escolhido não deve variar.

## 4. Construir a árvore

Agora a recursão. A partir de um grupo:

1. se ele já for puro, ou se um critério de parada for atingido, devolva uma folha com a classe mais frequente;
2. caso contrário, encontre o melhor corte, divida o grupo e repita dos dois lados.

Dois critérios de parada aparecem na aula e serão usados mais adiante: `max_depth`, a profundidade máxima, e `min_samples_leaf`, o número mínimo de exemplos em uma folha. Deixe os dois como parâmetros.

Represente a árvore do jeito que preferir. Uma forma simples é um dicionário: `{'leaf': classe, 'n': tamanho}` para folhas e `{'feature': ..., 'threshold': ..., 'left': ..., 'right': ..., 'n': ...}` para os nós internos. A função `print_tree` abaixo espera esse formato; se você usar outro, adapte-a.

In [ ]:
def build_tree(X, y, depth=0, max_depth=None, min_samples_leaf=1):
    # Caso base: grupo puro, profundidade esgotada ou nenhum corte possível
    # -> devolva {'leaf': classe mais frequente, 'n': len(y)}.
    # Caso geral: encontre o melhor corte, divida X e y em dois e chame
    # build_tree novamente de cada lado, com depth + 1.
    ...

In [ ]:
def print_tree(node, indent=''):
    if 'leaf' in node:
        print(f"{indent}--> {node['leaf']} ({node['n']})")
        return
    print(f"{indent}{node['feature']} <= {node['threshold']:.3f}?  (n={node['n']})")
    print_tree(node['left'], indent + '   ')
    print_tree(node['right'], indent + '   ')

tree = build_tree(X_train, y_train, max_depth=3)
print_tree(tree)

Leia a árvore impressa em voz alta, de cima para baixo, como uma sequência de perguntas. Compare com as regras que você escreveu na semana passada: quais atributos o algoritmo usou? Ele usou algum que você tinha descartado? Ignorou algum que você considerava essencial?

**Minhas observações:** ...

## 5. Prever e avaliar

Para classificar uma fruta, percorra a árvore a partir da raiz, seguindo para a esquerda quando a resposta for sim.

In [ ]:
def predict_one(node, row):
    """row é uma linha do DataFrame (uma Series com os atributos)."""
    # Desça a partir de node até chegar a uma folha, seguindo para a esquerda
    # quando row[feature] <= threshold.
    ...


def predict(node, X):
    return X.apply(lambda row: predict_one(node, row), axis=1)

In [ ]:
y_pred = predict(tree, X_train)
accuracy = (y_pred == y_train).mean()
print(f'Acurácia no treino: {accuracy:.1%}')

## 6. Métricas

A acurácia é um número só, e um número só esconde muita coisa. Implemente as métricas da aula para ver o que está por baixo.

A matriz de confusão cruza a classe real (linhas) com a classe prevista (colunas). A diagonal são os acertos.

In [ ]:
def confusion_matrix(y_true, y_pred):
    """Devolve um DataFrame com uma linha e uma coluna por classe."""
    # Comece com uma tabela de zeros indexada pelas classes e percorra os pares
    # (real, previsto), somando 1 na célula correspondente.
    ...


confusion_matrix(y_train, y_pred)

Para cada classe, tratamos aquela classe como positiva e todas as outras como negativas. Daí saem os quatro números da aula e as três métricas:

- precisão = TP / (TP + FP) — dos que o modelo apontou como desta classe, quantos eram;
- revocação = TP / (TP + FN) — dos que eram desta classe, quantos o modelo encontrou;
- F1 = 2 · P · R / (P + R).

Cuidado com o denominador zero: ele acontece de verdade neste conjunto.

In [ ]:
def class_report(y_true, y_pred):
    """Devolve um DataFrame com precisão, revocação, F1 e suporte por classe."""
    # Para cada classe, conte TP, FP e FN tratando-a como positiva e as demais
    # como negativas. A matriz de confusão já tem esses números.
    ...


class_report(y_train, y_pred)

Olhe a tabela com atenção. Com `max_depth=3`, pelo menos uma classe deve ter revocação zero: a árvore não criou nenhuma folha para ela.

**Responda:**

1. Quais classes o modelo nunca prevê? O que elas têm em comum?
2. A acurácia total deixava isso visível?
3. Por que o algoritmo abandonou essas classes? Lembre que o custo de um corte é ponderado pelo tamanho dos grupos.

**Minhas respostas:** ...

## 7. Profundidade, poda e overfitting

Agora varie os critérios de parada e observe o que muda. Construa árvores com profundidades diferentes e sem limite nenhum, e compare:

- a acurácia no treino;
- o número de folhas;
- a revocação das classes raras.

Monte uma tabela com esses resultados.

In [ ]:
def count_leaves(node):
    return 1 if 'leaf' in node else count_leaves(node['left']) + count_leaves(node['right'])


# Compare as profundidades. Inclua max_depth=None.
...

A árvore sem limite acerta tudo no treino. Isso não basta, por si só, para dizer que ela é melhor que as outras.

**Responda:** por que medir a qualidade da árvore nos mesmos dados que a construíram não ajuda a escolher a profundidade? Que dados usaríamos para essa escolha?

**Minha resposta:** ...

### Escolhendo a profundidade com validação

Precisamos de uma estimativa obtida fora dos dados usados na construção. Uma forma, viável com um conjunto pequeno, é a validação *leave-one-out*: para cada fruta, construa a árvore com as outras 55 e preveja aquela que ficou de fora.

Implemente essa avaliação e use-a para escolher a profundidade. Compare as métricas por classe, não só a acurácia.

Depois olhe a tabela e responda à pergunta que realmente importa: **a poda ajudou neste conjunto?**

Ela é aberta de verdade, e a resposta pode ser não. São 56 frutas; uma diferença de duas ou três previsões entre duas árvores não distingue modelo nenhum. Se os números não sustentarem a poda, diga isso — e escolha sua árvore final por outro critério, como o tamanho ou a facilidade de explicá-la. Concluir que a poda ajudou porque *deveria* ajudar é exatamente o erro que esta etapa existe para evitar.

In [ ]:
def leave_one_out(X, y, **params):
    """Devolve as previsões, uma por fruta, cada uma feita por uma árvore
    construída sem aquela fruta."""
    # Para cada índice i: construa a árvore com X.drop(i) e y.drop(i),
    # preveja X.loc[i] e guarde o resultado.
    ...


# Use leave_one_out para comparar as profundidades e escolher uma.
...

**Profundidade escolhida:** ...

**Por quê:** ...

## 8. Um experimento com o desbalanceamento

As classes raras continuam difíceis, e a aula apresentou algumas saídas: pesos de classe, reamostragem e ajuste do limiar de decisão.

Experimente **uma** delas. A mais direta de implementar é o *oversampling*: replique os exemplos das classes raras até que elas tenham mais peso na construção da árvore, e refaça a avaliação.

Uma observação importante vale para qualquer estratégia: altere apenas o conjunto de treino. Avaliar em dados reamostrados mede o modelo em uma distribuição que não existe.

**Responda:** a revocação das classes raras melhorou? Alguma outra métrica piorou junto?

**Minha resposta:** ...

In [ ]:
...

## 9. Congelar o modelo e prever no teste

Escolha uma única árvore final, com a profundidade e a estratégia de desbalanceamento que você decidiu. Registre a escolha antes de abrir o teste e não a altere depois.

**Minha árvore final:** ...

**Estimativa de acertos no teste:** ... de 15.

In [ ]:
TEST_URL = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/frutas/test_extended.csv'
test = pd.read_csv(TEST_URL)
X_test = test[FEATURES]
test

In [ ]:
final_tree = ...  # a árvore que você congelou

test_predictions = test[['id']].copy()
test_predictions['prediction'] = predict(final_tree, X_test)
test_predictions.to_csv('tree_predictions.csv', index=False)
display(test_predictions)

try:
    from google.colab import files
    files.download('tree_predictions.csv')
except ImportError:
    print('O arquivo tree_predictions.csv foi salvo na pasta atual.')

## 10. Comparar com o k-NN (recomendado)

Esta seção é opcional no sentido de que a entrega é aceita sem ela. Mas é aqui que está o resultado mais interessante da atividade, e quem parar antes perde a melhor pergunta da semana: **por que dois algoritmos corretos, treinados nos mesmos dados, discordam tanto?**

O k-NN não constrói regras: guarda os exemplos e, para classificar uma fruta nova, procura as `k` mais parecidas e escolhe a classe mais votada entre elas.

Implemente-o e avalie com a mesma validação leave-one-out que você usou para a árvore. Três pontos merecem atenção:

- a distância euclidiana soma as diferenças de todos os atributos, e a massa varia de 76 a 448 enquanto a largura varia de 5,4 a 10,4. Sem normalizar, a massa decide quase sozinha. Compare os resultados com e sem a normalização min-max;
- teste valores diferentes de `k` e observe o efeito nas classes raras. Com `k = 5`, uma classe de 3 exemplos consegue vencer alguma votação? Não responda só sim ou não: **imprima os votos dos vizinhos** e veja com quantos ela ganha, e contra o quê;
- se um dos dois modelos vencer com folga, explique por quê, olhando a forma dos dados. A árvore só corta paralelo aos eixos, uma pergunta por atributo de cada vez; o k-NN enxerga vizinhanças em qualquer direção. Que formato os agrupamentos de frutas têm no espaço de atributos, e qual dos dois esse formato favorece?

Compare também o tipo de explicação que cada modelo oferece para uma mesma previsão.

In [ ]:
...

## 11. Avaliação depois da liberação do gabarito

Cole abaixo o link direto fornecido pelo professor. Execute esta seção somente depois de congelar suas previsões.

Além de avaliar a sua árvore final, aproveite para aplicar ao teste **também a árvore sem limite** e comparar. Agora você tem os dois números para cada uma: o da validação e o do teste. Eles contam a mesma história?

In [ ]:
ANSWER_KEY_URL = ''  # Cole aqui o link direto liberado pelo professor.

if not ANSWER_KEY_URL.strip():
    print('O gabarito ainda não foi informado. Preserve suas previsões e volte depois.')
else:
    answer_key = pd.read_csv(ANSWER_KEY_URL)
    results = test_predictions.merge(answer_key, on='id', validate='one_to_one')
    results = results.rename(columns={'fruit_name': 'y_true', 'prediction': 'y_pred'})
    results['correct'] = results['y_pred'] == results['y_true']

    print(f"Acurácia no teste: {results['correct'].mean():.1%}")
    display(confusion_matrix(results['y_true'], results['y_pred']))
    display(class_report(results['y_true'], results['y_pred']))
    display(results.loc[~results['correct']])

## 12. Fechamento

Responda com suas palavras:

1. Compare as perguntas que a árvore escolheu com as regras que você escreveu à mão na semana passada. Ela usou os mesmos atributos? Chegou a algum corte que não teria lhe ocorrido?
2. O resultado no teste ficou acima ou abaixo da sua estimativa? E em relação ao treino?
3. Quais confusões aparecem fora da diagonal da matriz? Elas são entre frutas parecidas?
4. As classes raras foram previstas no teste? A estratégia de desbalanceamento que você escolheu ajudou?
5. A poda ajudou? Responda com os números da validação e os do teste, e diga se os dois concordam.
6. Escolha uma previsão errada e explique, percorrendo a árvore, por que o modelo chegou a ela. Se sua árvore acertou todo o teste, use um erro da validação — eles não faltam. Uma decisão explicável é por isso mesmo uma decisão correta?
7. Se este fosse um problema de triagem médica, em vez de frutas, qual erro você preferiria cometer: um falso positivo ou um falso negativo? Que métrica você acompanharia?